# Project Parcel — Complete 3-Day Build (Synthetic Data)

> **Data provenance:** This notebook generates synthetic shipment, carrier, and warehouse data. It does not use real company shipment records. All costs and operational capacity assumptions are illustrative and must be replaced with measured business inputs before deployment.


In [1]:
# Cell 1 — Install and load the reusable project code
!pip -q install pandas numpy scikit-learn joblib matplotlib pytest

from google.colab import files
from zipfile import ZipFile
import io, sys, os, json
import numpy as np
import pandas as pd

uploaded = files.upload()  # Upload Project_Parcel_Synthetic_Colab.zip
zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if zip_name is None:
    raise ValueError("Please upload Project_Parcel_Synthetic_Colab.zip")
with ZipFile(io.BytesIO(uploaded[zip_name])) as zf:
    zf.extractall("/content")
sys.path.insert(0, "/content/Project_Parcel_Synthetic_Colab/src")

from parcel_solution import (
    Config, generate_synthetic_data, audit_dataframe, prepare_labels,
    build_feature_list, temporal_split, compare_models, evaluate_model,
    segment_error_analysis, explain_one, monitor_batch, save_bundle,
    load_bundle, predict_new
)
print("Project Parcel code loaded.")

Saving Project_Parcel_Final_Refined_Bundle.zip to Project_Parcel_Final_Refined_Bundle.zip
Project Parcel code loaded.


## Day 1 — Understand the problem and data

### 1. Generate the three synthetic datasets
The generator uses a fixed random seed and creates 8,000 shipments, 8 carriers, and 6 warehouses. It injects selected missing values and messy category formatting so that data auditing is meaningful.

In [2]:
shipments, carriers, warehouses = generate_synthetic_data(
    n=8000, seed=42, output_dir="/content/parcel_data"
)
print("Shipments:", shipments.shape)
print("Carriers:", carriers.shape)
print("Warehouses:", warehouses.shape)
display(shipments.head())
display(carriers)
display(warehouses)

Shipments: (8000, 13)
Carriers: (8, 5)
Warehouses: (6, 4)


,shipment_id,ship_date,promised_delivery_date,actual_delivery_date,carrier_id,warehouse_id,distance_km,package_weight_kg,declared_value,origin_region,destination_region,parcel_type,service_level
0,S0000000,2023-03-07 00:00:00+00:00,2023-03-09 00:00:00+00:00,2023-03-08 00:00:00+00:00,C06,W06,591.2,27.45,14.94,Central,South,medium_box,express
1,S0000001,2024-07-18 00:00:00+00:00,2024-07-22 00:00:00+00:00,2024-07-21 00:00:00+00:00,C06,W04,355.2,5.30,7.72,North,West,large_box,standard
2,S0000002,2024-04-22 00:00:00+00:00,2024-04-26 00:00:00+00:00,2024-04-26 00:00:00+00:00,C03,W01,212.6,4.23,38.92,North,North,large_box,standard
3,S0000003,2023-11-17 00:00:00+00:00,2023-11-24 00:00:00+00:00,2023-11-27 00:00:00+00:00,C03,W06,454.0,4.35,31.08,East,Central,document,economy
4,S0000004,2023-11-13 00:00:00+00:00,2023-11-17 00:00:00+00:00,2023-11-17 00:00:00+00:00,C08,W06,1245.7,1.84,22.08,Central,Central,large_box,standard


,carrier_id,carrier_name,carrier_tier,carrier_historical_ontime_rate,carrier_capacity_index
0,C01,SwiftShip,premium,0.95,1.15
1,C02,NorthStar,standard,0.88,1.00
2,C03,BudgetFreight,economy,0.76,0.82
3,C04,PrimeRoute,standard,0.86,0.95
4,C05,RapidParcel,premium,0.96,1.20
5,C06,ValueCarrier,economy,0.74,0.78
6,C07,MetroLogistics,standard,0.84,0.92
7,C08,RoadRunner,economy,0.80,0.85


,warehouse_id,warehouse_region,daily_capacity,planned_load_ratio
0,W01,North,1800,0.72
1,W02,South,1500,0.88
2,W03,East,1200,0.93
3,W04,West,1600,0.68
4,W05,Central,1400,0.81
5,W06,North,1100,0.97


### 2. Define the late label and prediction point

**Late definition:** a shipment is late when actual delivery is strictly after promised delivery.

**Prediction point:** dispatch time. Therefore actual delivery timestamp is label-only and must never be a model feature.

In [3]:
CONFIG = Config(
    shipment_id="shipment_id",
    shipped_at="ship_date",
    promised_at="promised_delivery_date",
    delivered_at="actual_delivery_date",
    top_k_fraction=0.10,
    false_negative_cost=5.0,  # illustrative assumption
    false_positive_cost=1.0   # illustrative assumption
)

audit = audit_dataframe(shipments, CONFIG)
display(audit)
print("Duplicate shipment IDs:", shipments["shipment_id"].duplicated().sum())
print("Late label definition: actual_delivery_date > promised_delivery_date")

,issue,count,decision
0,missing_distance_km,200,Retain and handle predictor missingness inside...
1,missing_package_weight_kg,144,Retain and handle predictor missingness inside...
2,missing_declared_value,160,Retain and handle predictor missingness inside...
3,missing_destination_region,96,Retain and handle predictor missingness inside...


Duplicate shipment IDs: 0
Late label definition: actual_delivery_date > promised_delivery_date


### 3. Join carrier and warehouse reference data

Check key uniqueness and validate many-to-one joins. In this synthetic exercise, the carrier and warehouse attributes are treated as known at dispatch. A real system would require time-stamped snapshots.

In [8]:
assert carriers["carrier_id"].is_unique
assert warehouses["warehouse_id"].is_unique

data = shipments.merge(
    carriers, on="carrier_id", how="left", validate="many_to_one"
)
data = data.merge(
    warehouses, on="warehouse_id", how="left", validate="many_to_one"
)
assert len(data) == len(shipments), "Unexpected row multiplication after joins"
print("Joined table:", data.shape)
print("Unmatched carrier rows:", data["carrier_tier"].isna().sum())
print("Unmatched warehouse rows:", data["daily_capacity"].isna().sum())
display(data.head())

Joined table: (8000, 20)
Unmatched carrier rows: 0
Unmatched warehouse rows: 0


,shipment_id,ship_date,promised_delivery_date,actual_delivery_date,carrier_id,warehouse_id,distance_km,package_weight_kg,declared_value,origin_region,destination_region,parcel_type,service_level,carrier_name,carrier_tier,carrier_historical_ontime_rate,carrier_capacity_index,warehouse_region,daily_capacity,planned_load_ratio
0,S0000000,2023-03-07 00:00:00+00:00,2023-03-09 00:00:00+00:00,2023-03-08 00:00:00+00:00,C06,W06,591.2,27.45,14.94,Central,South,medium_box,express,ValueCarrier,economy,0.74,0.78,North,1100,0.97
1,S0000001,2024-07-18 00:00:00+00:00,2024-07-22 00:00:00+00:00,2024-07-21 00:00:00+00:00,C06,W04,355.2,5.30,7.72,North,West,large_box,standard,ValueCarrier,economy,0.74,0.78,West,1600,0.68
2,S0000002,2024-04-22 00:00:00+00:00,2024-04-26 00:00:00+00:00,2024-04-26 00:00:00+00:00,C03,W01,212.6,4.23,38.92,North,North,large_box,standard,BudgetFreight,economy,0.76,0.82,North,1800,0.72
3,S0000003,2023-11-17 00:00:00+00:00,2023-11-24 00:00:00+00:00,2023-11-27 00:00:00+00:00,C03,W06,454.0,4.35,31.08,East,Central,document,economy,BudgetFreight,economy,0.76,0.82,North,1100,0.97
4,S0000004,2023-11-13 00:00:00+00:00,2023-11-17 00:00:00+00:00,2023-11-17 00:00:00+00:00,C08,W06,1245.7,1.84,22.08,Central,Central,large_box,standard,RoadRunner,economy,0.80,0.85,North,1100,0.97


In [17]:
# Normalize categorical columns before label creation and model training

categorical_cols = [
    "carrier_id",
    "warehouse_id",
    "carrier_tier",
    "warehouse_region",
    "origin_region",
    "destination_region",
    "parcel_type",
    "service_level"
]

for col in categorical_cols:
    if col in data.columns:
        data[col] = (
            data[col]
            .astype("string")
            .str.strip()
            .str.lower()
        )

print("Categorical normalization completed.")

# Verify service-level categories
print("\nService level categories:")
print(data["service_level"].value_counts(dropna=False))

Categorical normalization completed.

Service level categories:
service_level
standard    4388
economy     1824
express     1788
Name: count, dtype: Int64


### 4. Label creation, class balance and temporal split

A random split can mix historical and future operating patterns. We use oldest 70% for training, next 15% for validation, and latest 15% for final testing.

In [18]:
labelled = prepare_labels(data, CONFIG)
display(labelled["is_late"].value_counts().rename(index={0:"on_time", 1:"late"}))
display(labelled.groupby(labelled["ship_date"].dt.to_period("Q"))["is_late"].agg(["count","mean"]))

train_df, valid_df, test_df = temporal_split(labelled, CONFIG)
features = build_feature_list(labelled, CONFIG)
print("Features:", features)
print("Train / validation / test:", len(train_df), len(valid_df), len(test_df))
print("Train period:", train_df.ship_date.min(), "to", train_df.ship_date.max())
print("Validation period:", valid_df.ship_date.min(), "to", valid_df.ship_date.max())
print("Test period:", test_df.ship_date.min(), "to", test_df.ship_date.max())
assert CONFIG.delivered_at not in features and "is_late" not in features

,count
is_late,
on_time,5396
late,2604


/tmp/ipykernel_9001/1123662083.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  display(labelled.groupby(labelled["ship_date"].dt.to_period("Q"))["is_late"].agg(["count","mean"]))


,count,mean
ship_date,,
2023Q1,988,0.303644
2023Q2,1036,0.303089
2023Q3,1027,0.299903
2023Q4,1012,0.397233
2024Q1,1020,0.332353
2024Q2,937,0.311633
2024Q3,1018,0.281925
2024Q4,962,0.376299


Features: ['distance_km', 'package_weight_kg', 'declared_value', 'carrier_historical_ontime_rate', 'carrier_capacity_index', 'daily_capacity', 'planned_load_ratio', 'carrier_id', 'warehouse_id', 'carrier_tier', 'warehouse_region', 'origin_region', 'destination_region', 'parcel_type', 'service_level']
Train / validation / test: 5600 1200 1200
Train period: 2023-01-01 00:00:00+00:00 to 2024-05-23 00:00:00+00:00
Validation period: 2024-05-23 00:00:00+00:00 to 2024-09-09 00:00:00+00:00
Test period: 2024-09-09 00:00:00+00:00 to 2024-12-30 00:00:00+00:00


In [20]:
# Fix pandas nullable string values for scikit-learn compatibility

import numpy as np
import pandas as pd

for df in [train_df, valid_df]:
    for col in features:
        if col in df.columns and df[col].dtype.name == "string":
            df[col] = df[col].astype(object)
            df[col] = df[col].where(df[col].notna(), np.nan)

print("String dtype compatibility fix applied.")

# Verify that no pandas nullable string columns remain
print("Training string columns:",
      train_df[features].select_dtypes(include=["string"]).columns.tolist())

print("Validation string columns:",
      valid_df[features].select_dtypes(include=["string"]).columns.tolist())

String dtype compatibility fix applied.
Training string columns: []
Validation string columns: []


## Day 2 — Build and validate models

### 5. Compare two meaningfully different models
Logistic Regression is the linear, interpretable baseline. Random Forest can capture nonlinear relationships and interactions. Both are trained on the earlier time period and evaluated on the same later validation period. We select the candidate using **validation PR-AUC**, because late shipments are the positive class and the operations team needs a useful ranking under limited review capacity. The final test period is held out and is not used for model selection.


In [21]:
fitted, comparison = compare_models(train_df, valid_df, features, CONFIG)
display(comparison)
best_name = comparison.iloc[0]["model"]
best_model = fitted[best_name]
print("Selected model based on validation PR-AUC:", best_name)

,model,n,late_rate,pr_auc,roc_auc,top_k_fraction,top_k_count,top_k_precision,top_k_recall,top_k_f1,brier
0,logistic_regression,1200,0.291667,0.531999,0.714124,0.1,120,0.658333,0.225714,0.336170,0.217288
1,random_forest,1200,0.291667,0.495975,0.698128,0.1,120,0.591667,0.202857,0.302128,0.198286


Selected model based on validation PR-AUC: logistic_regression


#### Model selection result

The validation results select **Logistic Regression**: PR-AUC **0.5320**, compared with **0.4960** for Random Forest. Logistic Regression also has higher validation top-10% precision (**65.83%** vs **59.17%**) and top-10% recall (**22.57%** vs **20.29%**). Its ROC-AUC is also higher (**0.7141** vs **0.6981**). Random Forest has a lower Brier score (**0.1983** vs **0.2173**), so it has better probability calibration by this metric in this run; however, the primary selection criterion is validation PR-AUC and the operational ranking quality. This is a validation-based choice, not a claim that Logistic Regression is universally better.


### 6. Business-oriented evaluation

Top-10% metrics assume operations can review 10% of shipments. False-negative cost 5 and false-positive cost 1 are illustrative assumptions, not supplied company costs. Do not tune the model using final test results.

In [23]:
import pandas as pd
import numpy as np

# Fix pandas nullable values in all three datasets
for df in [train_df, valid_df, test_df]:

    for col in features:
        if col not in df.columns:
            continue

        # Convert nullable string columns to regular Python objects
        if isinstance(df[col].dtype, pd.StringDtype):
            df[col] = df[col].astype(object)

        # Replace pandas NA with NumPy NaN
        if df[col].dtype == object:
            df[col] = df[col].where(df[col].notna(), np.nan)

print("Data compatibility check completed.")

for name, df in [
    ("Train", train_df),
    ("Validation", valid_df),
    ("Test", test_df)
]:
    print(f"\n{name} dataset:")
    print(df[features].dtypes)

Data compatibility check completed.

Train dataset:
distance_km                       float64
package_weight_kg                 float64
declared_value                    float64
carrier_historical_ontime_rate    float64
carrier_capacity_index            float64
daily_capacity                      int64
planned_load_ratio                float64
carrier_id                         object
warehouse_id                       object
carrier_tier                       object
warehouse_region                   object
origin_region                      object
destination_region                 object
parcel_type                        object
service_level                      object
dtype: object

Validation dataset:
distance_km                       float64
package_weight_kg                 float64
declared_value                    float64
carrier_historical_ontime_rate    float64
carrier_capacity_index            float64
daily_capacity                      int64
planned_load_ratio             

In [24]:
valid_metrics, valid_threshold, valid_prob = evaluate_model(
    best_model, valid_df, features, CONFIG
)
test_metrics, test_threshold, test_prob = evaluate_model(
    best_model, test_df, features, CONFIG
)
print("VALIDATION METRICS")
print(json.dumps(valid_metrics, indent=2))
print("Validation threshold under illustrative costs:", valid_threshold)
print("\nFINAL HELD-OUT TEST METRICS")
print(json.dumps(test_metrics, indent=2))

VALIDATION METRICS
{
  "n": 1200,
  "late_rate": 0.2916666666666667,
  "pr_auc": 0.5319986426705021,
  "roc_auc": 0.7141243697478992,
  "top_k_fraction": 0.1,
  "top_k_count": 120,
  "top_k_precision": 0.6583333333333333,
  "top_k_recall": 0.2257142857142857,
  "top_k_f1": 0.33617021276595743,
  "brier": 0.2172876018949101
}
Validation threshold under illustrative costs: {'threshold': 0.29, 'cost': 759.0, 'tn': 151, 'fp': 699, 'fn': 12, 'tp': 338}

FINAL HELD-OUT TEST METRICS
{
  "n": 1200,
  "late_rate": 0.3591666666666667,
  "pr_auc": 0.5467648258959179,
  "roc_auc": 0.6826836914183303,
  "top_k_fraction": 0.1,
  "top_k_count": 120,
  "top_k_precision": 0.6583333333333333,
  "top_k_recall": 0.18329466357308585,
  "top_k_f1": 0.2867513611615245,
  "brier": 0.22461601864231265
}


#### Operational policy: review the highest-risk 10%

The illustrative cost ratio assigns a missed late parcel a cost of 5 and a false alarm a cost of 1. On validation, the cost-minimizing threshold was **0.29**, but it flags **1,037 of 1,200 parcels (86.4%)**: 699 false positives and 338 true positives. That exceeds the assumed capacity to review only 10% of shipments, so it is not an operationally feasible rule under the stated constraint. Instead, rank parcels by predicted risk and send the highest-risk **120 of 1,200 (10%)** for review. This is a capacity-based prioritization policy, not a claim that the top-10% rule minimizes the assumed cost.

On the held-out test set, the top 120 had **65.83% precision** and **18.33% recall**. Thus, roughly two-thirds of reviewed parcels were late, but the policy found fewer than one in five late parcels overall. Many late parcels remain unflagged; the queue should support targeted intervention, not replace broader service monitoring. Revisit the 10% capacity and cost ratio with operations data before deployment.

**Synthetic-data limitation:** all records are generated for this exercise. The cost weights (5:1) and 10% review capacity are assumptions, not observed company economics or an agreed service-level constraint. Metrics demonstrate the pipeline only and do not establish expected real-world savings or performance.


### 7. Break the model: segment error analysis

Inspect carrier, destination region, origin region, parcel type, and service level. Small segments are excluded by the analysis function.
Missing destination values are retained as an explicit `<MISSING>` segment so their error profile is visible rather than being mistaken for a real region.

In [25]:
segments = segment_error_analysis(
    test_df, test_prob,
    ["carrier_id", "destination_region", "origin_region", "parcel_type", "service_level"]
)
display(segments.sort_values(["top10_precision", "n"], ascending=[True, False]))

,segment_column,segment,n,late_rate,top10_precision
4,carrier_id,c05,145,0.179310,0.000000
0,carrier_id,c01,138,0.210145,0.000000
13,destination_region,<MISSING>,14,0.214286,0.000000
17,origin_region,south,240,0.375000,0.500000
3,carrier_id,c04,150,0.333333,0.500000
16,origin_region,north,251,0.338645,0.565217
9,destination_region,east,254,0.413386,0.566667
21,parcel_type,medium_box,378,0.365079,0.575000
25,service_level,standard,671,0.333830,0.611111
2,carrier_id,c03,176,0.494318,0.617647


### 8. Global feature importance and a single prediction

Permutation importance is measured on the held-out test set. It describes predictive reliance, not causation. The local explanation reports input facts and risk band, not a causal reason.

In [26]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model, test_df[features], test_df["is_late"],
    scoring="average_precision", n_repeats=5,
    random_state=CONFIG.random_state, n_jobs=-1
)
importance = pd.DataFrame({
    "feature": features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
display(importance)

one_parcel_explanation = explain_one(best_model, test_df.iloc[[0]], features)
print(json.dumps(one_parcel_explanation, indent=2))

,feature,importance_mean,importance_std
14,service_level,0.067105,0.005049
0,distance_km,0.041527,0.007646
3,carrier_historical_ontime_rate,0.024220,0.004739
4,carrier_capacity_index,0.018633,0.004488
6,planned_load_ratio,0.005167,0.004652
5,daily_capacity,0.003713,0.002769
7,carrier_id,0.002703,0.002601
9,carrier_tier,0.000536,0.001547
13,parcel_type,0.000399,0.001677
1,package_weight_kg,-0.000075,0.001578


{
  "risk_score": 0.5492271246175184,
  "risk_band": "medium",
  "input_facts": {
    "distance_km": "920.1",
    "package_weight_kg": "2.83",
    "declared_value": "42.18",
    "carrier_historical_ontime_rate": "0.88",
    "carrier_capacity_index": "1.0",
    "daily_capacity": "1200",
    "planned_load_ratio": "0.93",
    "carrier_id": "c02",
    "warehouse_id": "w03",
    "carrier_tier": "standard",
    "warehouse_region": "east",
    "origin_region": "north",
    "destination_region": "east",
    "parcel_type": "small_box",
    "service_level": "standard"
  },
  "caveat": "Input facts are not causal explanations. Use global held-out importance for model-level interpretation."
}


## Day 3 — Make it usable and trustworthy

### 9. Input drift check and model packaging

First compare the held-out batch with the training reference. Then deliberately shift a numeric feature to verify the monitor can detect an out-of-range distribution. The shifted batch is a test fixture only; it is not used for model training or reported as production drift.

In [27]:
baseline_drift_alerts = monitor_batch(
    test_df[features], features, train_df[features]
)
print("Held-out batch drift alerts:", baseline_drift_alerts)

# Deliberate monitoring test: move distance far beyond its training distribution.
shifted_batch = test_df[features].copy()
shifted_batch["distance_km"] = shifted_batch["distance_km"].fillna(0) + 5000
shift_test_alerts = monitor_batch(shifted_batch, features, train_df[features])
print("Deliberately shifted batch alerts:", shift_test_alerts)
assert any(a["feature"] == "distance_km" and a["type"] == "high_psi"
           for a in shift_test_alerts), "Drift monitor did not detect deliberate distance shift"

model_path = save_bundle(
    best_model, features, CONFIG, train_df,
    "/content/parcel_model.joblib"
)
print("Saved model:", model_path)

Held-out batch drift alerts: []
Deliberately shifted batch alerts: [{'feature': 'distance_km', 'type': 'high_psi', 'psi': 12.433088730615204}]
Saved model: /content/parcel_model.joblib


### 10. Score new parcels

For demonstration, use feature rows from the held-out period as unseen-to-training examples. No label or actual-delivery timestamp is passed to the prediction function.

In [28]:
new_parcels = test_df[features].head(20).copy()
bundle = load_bundle("/content/parcel_model.joblib")
predictions = predict_new(new_parcels, bundle)
display(predictions[["late_risk_probability", "risk_band"]].head(20))
predictions.to_csv("/content/parcel_predictions_demo.csv", index=False)
files.download("/content/parcel_predictions_demo.csv")
files.download("/content/parcel_model.joblib")

,late_risk_probability,risk_band
6800,0.549227,medium
6801,0.499765,medium
6802,0.479705,medium
6803,0.522065,medium
6804,0.750658,high
6805,0.432797,medium
6806,0.265736,low
6807,0.439101,medium
6808,0.741426,high
6809,0.329064,low


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 12. Interpretation and deployment limitations

The held-out test PR-AUC is **0.5468** and ROC-AUC is **0.6827**. Under the top-10% review policy, precision is **65.83%** and recall is **18.33%**. The model therefore enriches a small review queue for late shipments, but it misses most late shipments. Do not describe it as a comprehensive late-delivery detector.

The dataset is synthetic, so these figures are not evidence of real operational performance. Before any real deployment, validate on time-stamped company data, confirm every feature is available at dispatch, replace illustrative costs/capacity with measured values, calibrate risk scores, and monitor outcomes by carrier, region, season, and parcel type. In particular, confirm that carrier historical on-time rate and planned load ratio are calculated only from information available at the prediction time.


### 11. Automated smoke checks

In [29]:
assert len(train_df) + len(valid_df) + len(test_df) == len(labelled)
assert CONFIG.delivered_at not in features
assert "is_late" not in features
assert test_metrics["n"] == len(test_df)
assert set(labelled["is_late"].unique()).issubset({0, 1})
assert predictions["late_risk_probability"].between(0, 1).all()
print("All notebook smoke checks passed.")

All notebook smoke checks passed.
